In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [7]:
load_dotenv()

True

# lets build a AI company brouchure builder

1. parse all the urls from a website/webpage

In [32]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_all_links(url):
    try:
        # Send an HTTP GET request with a User-Agent to avoid being blocked
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        
        # Raise an exception for bad status codes (404, 500, etc.)
        response.raise_for_status()
        
        # Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        links = []
        # Find all <a> tags with an 'href' attribute
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            # Convert relative URLs (e.g., "/about") to absolute URLs
            full_url = urljoin(url, href)
            links.append(full_url)
            
        return links

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return []


In [33]:
target_url = "https://www.bmw.in/en/index.html"
all_links = get_all_links(target_url)

print(f"Found {len(all_links)} links:")
for link in all_links:
    print(link)

Found 142 links:
https://www.bmw.in/en/index.html#main
https://www.bmw.in/en/index.html
https://www.bmw.in/en/all-models.html
https://www.bmw.in/en/topics/Fascination-BMW/electromobility2020-new/electric-cars.html
https://www.bmw.in/en/shop/ls/cart/digital-services
https://www.bmw.in/en/shop/ls/wishlist/login
https://www.bmw.in/dlo/en-in/bmw/dealer-locator.html
https://www.bmw.in/en/index.html
https://www.bmw.in/dlo/en-in/bmw/dealer-locator.html
https://www.bmw.in/en-in/sl/stocklocator#/results?filters=is_installment=%27true%27
https://www.bmw.in/en/configurator.html
https://www.bmw.in/en/topics/special-offers.html
https://www.bmw.in/en/shop/ls/cp/connected-drive
https://www.bmw.in/en/bmw-alpina.html
https://www.bmw.in/en/bmw-alpina.html
https://www.bmw.in/en/bmw-financial-services.html
https://www.bmw.in/en/bmw-financial-services/bmw-loan.html
https://www.bmw.in/en/topics/offers-and-services/financial-services-2020/bmw-financial-service.html
https://www.bmw.in/en/fastlane/bmw-financia

2. filter out only useful links required for building brochure

In [35]:
from pydantic import BaseModel

In [36]:
class FilteredLinksBatch(BaseModel):
    useful_links: list[str]

In [41]:
def filter_links_with_llm(company_url, all_links):
    """
    Uses OpenAI to filter a raw list of URLs down to only the ones
    relevant for generating a corporate brochure.
    """
    print(f"Analyzing {len(all_links)} links with OpenAI...")
    
    prompt = f"""
    You are an AI assistant helping build a company brochure for {company_url}.
    Review the following list of URLs extracted from the company's website.
    Identify which links are highly relevant for gathering foundational corporate information, 
    such as company history, team, mission, products, services, solutions, case studies, or contact details.
    
    Exclude irrelevant links like social media sharing buttons (twitter, facebook, linkedin), 
    login pages, privacy policies, terms of service, RSS feeds, or duplicate anchor tags.
    
    Return a clean list containing ONLY the useful absolute URLs.
    
    List of URLs to analyze:
    {json.dumps(all_links, indent=2)}
    """
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a professional web data analyst."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format=FilteredLinksBatch
    )
    useful_links = response.choices[0].message.parsed.useful_links
    print(f"Found {len(useful_links)} useful links")
    return useful_links

In [42]:
useful_links = filter_links_with_llm(company_url="https://www.bmw.in/en/index.html", all_links=all_links)

Analyzing 142 links with OpenAI...
Found 13 useful links


3. hit the useful links, gather their content

In [44]:
import requests
from bs4 import BeautifulSoup

def scrape_content_from_links(links):
    """
    Loops through the filtered list of URLs, extracts the clean text 
    from each webpage, and combines them into one raw knowledge base.
    """
    combined_context = ""
    
    for link in links:
        try:
            print(f"Scraping content from: {link}")
            
            # Send GET request with a generic user agent to prevent basic bot blocks
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
            response = requests.get(link, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Remove scripts and style elements so we don't capture code or CSS
            for script_or_style in soup(["script", "style", "meta", "noscript"]):
                script_or_style.decompose()
                
            # Extract plain text from the webpage HTML
            text = soup.get_text(separator=' ')
            
            # Clean up whitespace, tabs, and newline gaps
            clean_text = " ".join(text.split())
            
            # Append to our master context with a clear header for the LLM
            combined_context += f"\n\n--- Source URL: {link} ---\n{clean_text}"
            
        except Exception as e:
            print(f"  [Error] Skipping {link}: {e}")
            
    return combined_context

In [45]:
try:
    print("Starting scraping pipeline...")
    raw_corporate_knowledge = scrape_content_from_links(useful_links)
    print("\n--- Scraping Complete! ---")
    print(f"Total characters gathered for context: {len(raw_corporate_knowledge)}")
    
    # Optional: Preview the first 500 characters to make sure it looks like clean text
    print("\nPreview of gathered text:")
    print(raw_corporate_knowledge[:500] + "...")
    
except NameError:
    print("Error: Please make sure you have executed Block 2 and that 'filtered_list' exists!")


Starting scraping pipeline...
Scraping content from: https://www.bmw.in/en/index.html
Scraping content from: https://www.bmw.in/en/all-models.html
Scraping content from: https://www.bmw.in/en/configurator.html
Scraping content from: https://www.bmw.in/en/topics/special-offers.html
Scraping content from: https://www.bmw.in/en/bmw-financial-services.html
Scraping content from: https://www.bmw.in/en/topics/offers-and-services/financial-services-2020/bmw-financial-service.html
Scraping content from: https://www.bmw.in/en/topics/owners/contact-us.html
Scraping content from: https://www.bmw.in/en/topics/Fascination-BMW/the-corporation-original/the-bmw-group.html
Scraping content from: https://www.bmw.in/en/topics/Fascination-BMW/the-corporation-original/bmw-india-pvt--ltd-.html
Scraping content from: https://www.bmw.in/en/more-bmw/customer-support.html
Scraping content from: https://www.bmw.in/en/more-bmw/bmw-company-car/sustainability.html
Scraping content from: https://www.bmw.in/en/fastla

4. build a brochure out of raw corporate knowledge

In [48]:
from IPython.display import display, Markdown

In [50]:
def generate_brochure(company_url, knowledge_base):
    """
    Sends the compiled web text data to OpenAI to generate 
    a highly structured, elegant Markdown corporate brochure.
    """
    print("Generating your professional brochure... (This may take a moment)")
    
    system_instruction = (
        "You are an elite corporate copywriter and marketing director. "
        "Your task is to generate a comprehensive, highly polished, and compelling "
        "corporate brochure based strictly on the provided context. Output the response "
        "in clean, perfectly formatted Markdown with an impact-driven layout."
    )
    
    user_prompt = f"""
    Create a detailed, publication-ready corporate brochure for the company at {company_url}.
    Use the raw webpage content below as your single source of truth. Extract key elements, 
    organize them logically, and make it engaging.
    
    Structure the brochure with the following sections:
    - **Header**: Company name, tagline, and an impact statement.
    - **Executive Summary / About Us**: Corporate identity, mission, vision, and core principles.
    - **Products, Services & Solutions**: Detailed breakdown of their main offerings and unique value propositions.
    - **Why Choose Us**: Key differentiators, achievements, or strengths extracted from the text.
    - **Contact Information**: Clear summary of addresses, key phone numbers, or corporate contact info.
    
    Raw context data:
    {knowledge_base}
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": system_instruction
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.7
    )
    content = response.choices[0].message.content
    return content

In [52]:

# Uses the target_url and raw_corporate_knowledge variables from previous blocks
final_markdown_brochure = generate_brochure("https://www.bmw.in/en/index.html", raw_corporate_knowledge)

print("\n--- Generation Complete! Rendering Document Below ---\n")
# This renders the Markdown beautifully inside the Jupyter Notebook cell interface
display(Markdown(final_markdown_brochure))

Generating your professional brochure... (This may take a moment)

--- Generation Complete! Rendering Document Below ---



# BMW India Corporate Brochure

---

## Header
**Company Name**: BMW India  
**Tagline**: "Sheer Driving Pleasure"  
**Impact Statement**: "Transforming mobility through innovation, luxury, and sustainability."

---

## Executive Summary / About Us

**Corporate Identity**:  
BMW India Pvt. Ltd. stands as a prominent player in the premium automotive market, offering a unique blend of luxury, performance, and technology.

**Mission**:  
To deliver extraordinary driving experiences through innovative automotive engineering and sustainable practices.

**Vision**:  
To be the leading premium automotive brand in India, renowned for excellence in quality, performance, and customer satisfaction.

**Core Principles**:
- **Sustainability**: Committed to reducing environmental impact through electric mobility and responsible resource management.
- **Innovation**: Continuously advancing automotive technology and smart services.
- **Customer Focus**: Prioritizing customer satisfaction and providing comprehensive support and services.

---

## Products, Services & Solutions

### Main Offerings:
1. **BMW Vehicle Range**:
   - **Sedans**: 2 Series, 3 Series, 5 Series, 7 Series, and M models.
   - **SUVs**: X1, X3, X5, X6, and X7.
   - **Electric Vehicles**: BMW iX1, i5, i7, and the innovative BMW i3.

2. **Financial Services**:
   - **BMW 360º Finance Plan**: Flexible financing options with lower monthly payments and assured buyback schemes.
   - **Leasing Options**: Tailored leasing solutions to fit individual needs.

3. **Digital Services**:
   - **BMW ConnectedDrive**: Innovative digital ecosystem connecting drivers to their vehicles and making mobility smarter.
   - **My BMW App**: A mobile application offering remote vehicle access, service booking, and real-time updates.

4. **Service & Maintenance**:
   - **Comprehensive Service Packages**: Including warranties, service cost estimators, and maintenance plans for peace of mind.

---

## Why Choose Us

**Key Differentiators**:
- **Luxury Experience**: Unmatched quality and attention to detail in every vehicle.
- **Sustainability Efforts**: Leading the charge in electric mobility and circular economy initiatives.
- **Global Presence**: A part of the BMW Group, one of the world's leading manufacturers of premium automobiles, with a strong global network and proven track record.
- **Support and Care**: Dedicated customer service with a three-step resolution process ensuring prompt assistance.

**Achievements**:
- Sold nearly 2.4 million vehicles globally in 2022, showcasing robust demand and trust in the BMW brand.
- Profit before tax of €23.5 billion on revenues of €142.6 billion in 2022, highlighting financial strength and operational excellence.

---

## Contact Information

**BMW India Customer Interaction Centre**:  
- **Phone**: 1800 102 2269  
- **Email**: contact.india@bmw.in  

**Address**:  
BMW Group India  
DLF Cyber City Phase II,  
2nd Floor, Oberoi Centre,  
Building No. 11,  
Gurgaon 122 002, India  

**Website**: [BMW India](https://www.bmw.in/en/index.html)

---

Stay connected with us for the latest updates and offerings. Experience the joy of driving with BMW!

--- 

*This brochure is designed to provide a comprehensive overview of BMW India’s offerings and corporate identity. For detailed terms and conditions, please consult your nearest BMW dealer.*

# final showdown

In [57]:
def main(company_url):
    try:
        all_links = get_all_links(company_url)
        useful_links = filter_links_with_llm(company_url="https://www.bmw.in/en/index.html", all_links=all_links)
        raw_corporate_knowledge = scrape_content_from_links(useful_links)
        final_markdown_brochure = generate_brochure("https://www.bmw.in/en/index.html", raw_corporate_knowledge)
        display(Markdown(final_markdown_brochure))
        return final_markdown_brochure
    except Exception as e:
        raise e

In [58]:
main(target_url)

Analyzing 142 links with OpenAI...
Found 29 useful links
Scraping content from: https://www.bmw.in/en/all-models.html
Scraping content from: https://www.bmw.in/en/topics/Fascination-BMW/electromobility2020-new/electric-cars.html
Scraping content from: https://www.bmw.in/dlo/en-in/bmw/dealer-locator.html
Scraping content from: https://www.bmw.in/en/configurator.html
Scraping content from: https://www.bmw.in/en/topics/special-offers.html
Scraping content from: https://www.bmw.in/en/bmw-alpina.html
Scraping content from: https://www.bmw.in/en/bmw-financial-services.html
Scraping content from: https://www.bmw.in/en/fastlane/bmw-financial-services-new/corporate-governance.html
Scraping content from: https://www.bmw.in/en/topics/owners/warranty-policy.html
Scraping content from: https://www.bmw.in/en/topics/owners/service-repair.html
Scraping content from: https://www.bmwindia.com/calculate-service?model=X-series-X1
Scraping content from: https://www.bmwindia.com/bmwrecall
Scraping content f

# BMW India Corporate Brochure

---

## Header
### **BMW India**
#### *Sheer Driving Pleasure*
**Experience the thrill of luxury driving with BMW, where innovation meets performance.**

---

## Executive Summary / About Us
### Corporate Identity
BMW India, part of the renowned BMW Group, stands as a leader in the premium automotive sector, committed to delivering unparalleled performance and luxury. 

### Mission
To provide innovative and sustainable mobility solutions while enhancing customer experiences through excellence in service and product offerings.

### Vision
To redefine the standard of premium automotive luxury and performance in India, fostering a sustainable future through electrification and technological advancements.

### Core Principles
- **Sustainability**: Commitment to eco-friendly practices across the production and supply chain.
- **Innovation**: Pioneering advancements in automotive technology, particularly in electric mobility.
- **Customer Focus**: Prioritizing customer needs and satisfaction through personalized experiences and services.

---

## Products, Services & Solutions
### **Main Offerings**
1. **BMW Electric Cars**
   - **i5**: A blend of executive comfort and electrifying performance.
   - **iX**: A cutting-edge SUV that redefines electric mobility with advanced features.
   - **i7**: Fully electric luxury sedan with multisensory entertainment.

2. **Series Vehicles**
   - **X Series**: Versatile SUVs designed for comfort and performance.
   - **7 Series**: The pinnacle of luxury, integrating advanced technology with superior craftsmanship.
   - **M Series**: Performance-driven models engineered for enthusiasts.

3. **Financial Solutions**
   - **BMW 360º Finance Plan**: Tailored financing options allowing flexibility and ease in ownership.
   - **Extended Warranty**: Up to 10 years of coverage ensuring peace of mind.

4. **Service & Maintenance**
   - **Service Inclusive**: Comprehensive service packages that cover all maintenance needs.
   - **BMW Joy Rewards**: Loyalty program offering discounts on services for older models.

5. **Connected Drive Services**
   - **BMW ConnectedDrive**: Seamless connectivity and control through smartphone integration.

---

## Why Choose Us
### Key Differentiators
- **Heritage**: Over a century of automotive excellence and innovation.
- **Quality Assurance**: Commitment to using only original BMW parts and expert service personnel.
- **Sustainability Focus**: Leading the charge in sustainable mobility with electric vehicle offerings and eco-friendly practices.
- **Customer-Centric Approach**: Personalized services and flexible financing options tailored to individual needs.

### Achievements
- Strong foothold in the luxury car segment with outstanding sales performance.
- Recognition for excellence in customer service and satisfaction.

---

## Contact Information
### **BMW India Headquarters**
- **Address**: DLF Cyber City Phase II, 2nd Floor, Oberoi Centre, Building No.11, Gurgaon, 122002, India.
- **Phone**: +91 124 4566 600
- **Email**: contact.india@bmw.in

### **Customer Support**
- **Toll-Free Number**: 1800 102 2269 (Mon-Sat, 09:00-18:30)
- **Roadside Assistance**: 1800 103 2211 (Available 24/7)

### **Dealership Locator**
Find your nearest authorized BMW dealer to explore our models and services: [Find a Dealer](https://www.bmw.in/en/find-a-dealer.html).

---

**Stay Connected with BMW India**
*For the latest updates and news, sign up at [BMW India](https://www.bmw.in/en/subscribe.html).*

---

*Copyright © BMW AG 2026. All rights reserved.*  
*Legal Disclaimer | Privacy Policy | Cookie Policy*

'# BMW India Corporate Brochure\n\n---\n\n## Header\n### **BMW India**\n#### *Sheer Driving Pleasure*\n**Experience the thrill of luxury driving with BMW, where innovation meets performance.**\n\n---\n\n## Executive Summary / About Us\n### Corporate Identity\nBMW India, part of the renowned BMW Group, stands as a leader in the premium automotive sector, committed to delivering unparalleled performance and luxury. \n\n### Mission\nTo provide innovative and sustainable mobility solutions while enhancing customer experiences through excellence in service and product offerings.\n\n### Vision\nTo redefine the standard of premium automotive luxury and performance in India, fostering a sustainable future through electrification and technological advancements.\n\n### Core Principles\n- **Sustainability**: Commitment to eco-friendly practices across the production and supply chain.\n- **Innovation**: Pioneering advancements in automotive technology, particularly in electric mobility.\n- **Custo